# Form Analyzer (TF pose sequences)

# Form Analyzer — TF temporal pose model (ST-GCN / 1D-CNN over 17-joint sequences)

Processes `training/process_form_data.py` output (`data/processed/forms/normalised.json`):
normalised 17-joint keypoint sequences + per-exercise quality labels
(`good` / `slight_misalignment` / `bad`). Matches the serving contract of
`app/form_analyzer_engine.py` (shoulder-width = 1, origin = hip midpoint).

In [ ]:
import importlib.util, os, pathlib, sys

# -- locate the Buddy-Up ai_service dir (training/ + data/ + notebooks/) from any CWD ----
p = pathlib.Path(os.getcwd()).resolve()
ai = None
while p != p.parent:
    for cand in (p / 'backend' / 'ai_service', p / 'ai_service', p):
        if (cand / 'training').is_dir() and (cand / 'data').is_dir() and (cand / 'notebooks').is_dir():
            ai = cand; break
    if ai is not None:
        break
    p = p.parent
if ai is None:
    raise RuntimeError("couldn't locate ai_service/ from cwd: " + os.getcwd())
sys.path.insert(0, str(ai / 'training'))
os.chdir(ai / 'notebooks')          # legacy ../data, ../models paths keep working

# -- install only what's missing (no-op inside the shared ml-env kernel) ----
_missing = [m for m in ['tensorflow', 'tf2onnx', 'onnxruntime'] if importlib.util.find_spec(m) is None]
if _missing:
    %pip install -q {" ".join(_missing)}
from tf_utils import set_memory_growth
set_memory_growth()
SCALE = os.environ.get("BUDDY_SCALE", "demo")


In [ ]:
# Load pose-keypoint sequences (extract from workout videos if missing/stale)
import os, sys, json, subprocess
from pathlib import Path
from collections import Counter

proc = Path('../data/processed/forms/normalised.json')
if not proc.exists() or os.environ.get('BUDDY_REPROCESS') == '1':
    print('Extracting pose keypoints from the workout-video datasets...')
    subprocess.run([sys.executable, '../training/process_workout_videos.py',
                    '--frames', '24'], check=True)

rows = json.loads(proc.read_text())
print('sequences:', len(rows))
classes = sorted({r['exercise'] for r in rows})
print('classes:', len(classes))
print('distribution:', dict(Counter(r['exercise'] for r in rows)))

In [ ]:
# Stratified train/val/test split + fixed-length padding (T, 17, 3)
import numpy as np
from sklearn.model_selection import train_test_split

T = {'smoke': 16, 'demo': 32, 'full': 48}[SCALE]
def pad(seq):
    out = np.zeros((T, 17, 3), dtype=np.float32)
    s = np.asarray(seq[:T], dtype=np.float32)
    out[:len(s)] = s
    return out

X = np.stack([pad(r['normalised']) for r in rows])
y = np.array([classes.index(r['exercise']) for r in rows], dtype=np.int32)

idx = np.arange(len(rows))
# Fallback to random split when any class has < 2 members (smoke uses 1/class)
from collections import Counter
min_per_class = min(Counter(r['exercise'] for r in rows).values())
strat = y if min_per_class >= 2 else None

train_idx, val_idx = train_test_split(idx, test_size=0.2, random_state=42, stratify=strat)
val_idx, test_idx = train_test_split(val_idx, test_size=0.5, random_state=42,
                                     stratify=strat[val_idx] if strat is not None else None)
print('train/val/test:', len(train_idx), len(val_idx), len(test_idx))

from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight('balanced', classes=np.unique(y[train_idx]), y=y[train_idx])
class_weight = dict(enumerate(cw))

In [ ]:
# Temporal model: per-joint Conv head -> BiLSTM over the sequence
import tensorflow as tf
from tf_utils import set_memory_growth
set_memory_growth()

inp = tf.keras.Input(shape=(T, 17, 3))
x = tf.keras.layers.Conv2D(32, (1, 3), padding='same', activation='relu')(inp)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Conv2D(64, (1, 1), padding='same', activation='relu')(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Reshape((T, 17 * 64))(x)
x = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True))(x)
x = tf.keras.layers.GlobalAveragePooling1D()(x)
x = tf.keras.layers.Dense(64, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
out = tf.keras.layers.Dense(len(classes), activation='softmax')(x)
m = tf.keras.Model(inp, out)
m.compile(tf.keras.optimizers.Adam(1e-3), 'sparse_categorical_crossentropy',
          metrics=['accuracy'])
m.summary()

In [ ]:
# Train (early-stopped) with class weights + LR decay
import time
EPOCHS = {'smoke': 2, 'demo': 30, 'full': 60}[SCALE]
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5,
                                     restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                         patience=2, min_lr=1e-5),
    tf.keras.callbacks.TensorBoard(log_dir=f'../logs/form_analyzer-{SCALE}'),
]
t0 = time.time()
m.fit(X[train_idx], y[train_idx], epochs=EPOCHS, batch_size=16,
      validation_data=(X[val_idx], y[val_idx]),
      class_weight=class_weight, callbacks=callbacks, verbose=1)
print(f'total {time.time()-t0:.0f}s')

In [ ]:
# Evaluate on the untouched test split + overfit gauge
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

p_test = m.predict(X[test_idx], verbose=0).argmax(1)
acc = float(accuracy_score(y[test_idx], p_test))
bal = float(balanced_accuracy_score(y[test_idx], p_test))
print(f'test n={len(test_idx)} acc={acc:.3f} balanced_acc={bal:.3f}')
print('confusion (rows=truth, cols=pred):')
print(confusion_matrix(y[test_idx], p_test))

p_train = m.predict(X[train_idx], verbose=0).argmax(1)
train_acc = float(accuracy_score(y[train_idx], p_train))
print(f'overfit gauge: train acc={train_acc:.3f} | test acc={acc:.3f} | gap={train_acc - acc:+.3f}')

In [ ]:
# Export ONNX (+ INT8) + persist the joint spec + log the run
from pathlib import Path
from tf_utils import export_keras_onnx, quantize_dynamic_onnx, mlflow_log

onnx = export_keras_onnx(m, Path('../models'), 'form_analyzer', '1.0.0',
    input_signature=[tf.TensorSpec((None, T, 17, 3), tf.float32, name='pose_input')])
q = quantize_dynamic_onnx(onnx)
mlflow_log({'name': 'form_analyzer', 'version': '1.0.0',
            'artifact_path': str(q), 'framework': 'tensorflow',
            'scenario': 'workout_videos', 'n_classes': len(classes),
            'metrics': {'test_accuracy': acc, 'balanced_accuracy': bal,
                        'train_accuracy': train_acc}})
print('exported', q)